In [1]:
import pandas as pd
import numpy as np

# ---------------------------
# Step 1: Create Dataset
# ---------------------------
data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast',
                'Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],

    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool',
                    'Mild','Cool','Mild','Mild','Mild','Hot','Mild'],

    'Humidity': ['High','High','High','High','Normal','Normal','Normal',
                 'High','Normal','Normal','Normal','High','Normal','High'],

    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong',
             'Weak','Weak','Weak','Strong','Strong','Weak','Strong'],

    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes',
                   'No','Yes','Yes','Yes','Yes','Yes','No']
}

df = pd.DataFrame(data)

# ---------------------------
# Step 2: Entropy Function
# ---------------------------
def entropy(target_col):
    values, counts = np.unique(target_col, return_counts=True)
    entropy_value = 0
    for i in range(len(values)):
        probability = counts[i] / np.sum(counts)
        entropy_value += -probability * np.log2(probability)
    return entropy_value

# ---------------------------
# Step 3: Information Gain
# ---------------------------
def information_gain(data, feature, target="PlayTennis"):

    total_entropy = entropy(data[target])

    values, counts = np.unique(data[feature], return_counts=True)

    weighted_entropy = 0
    for i in range(len(values)):
        subset = data[data[feature] == values[i]]
        weighted_entropy += (counts[i] / np.sum(counts)) * entropy(subset[target])

    return total_entropy - weighted_entropy

# ---------------------------
# Step 4: ID3 Algorithm
# ---------------------------
def ID3(data, features, target="PlayTennis"):

    # If all samples same class → return class
    if len(np.unique(data[target])) == 1:
        return np.unique(data[target])[0]

    # If no features left → return majority class
    if len(features) == 0:
        return data[target].mode()[0]

    # Compute information gain for all features
    gains = [information_gain(data, feature, target) for feature in features]

    # Select best feature
    best_feature = features[np.argmax(gains)]

    tree = {best_feature: {}}

    remaining_features = [f for f in features if f != best_feature]

    # Create subtree for each value
    for value in np.unique(data[best_feature]):
        subset = data[data[best_feature] == value]
        subtree = ID3(subset, remaining_features, target)
        tree[best_feature][value] = subtree

    return tree

# ---------------------------
# Step 5: Prediction Function
# ---------------------------
def predict(query, tree):

    for key in query:
        if key in tree:
            value = query[key]
            result = tree[key].get(value)

            if isinstance(result, dict):
                return predict(query, result)
            else:
                return result

# ---------------------------
# Step 6: Build Decision Tree
# ---------------------------
features = list(df.columns[:-1])
decision_tree = ID3(df, features)

print("Generated Decision Tree:")
print(decision_tree)

# ---------------------------
# Step 7: Classify New Sample
# ---------------------------
new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

prediction = predict(new_sample, decision_tree)

print("\nNew Sample:", new_sample)
print("Prediction:", prediction)

Generated Decision Tree:
{'Outlook': {'Overcast': 'Yes', 'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}}, 'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Prediction: No


In [2]:
import numpy as np
import pandas as pd

# ----------------------------------
# Step 1: Create Regression Dataset
# ----------------------------------
data = {
    'Size': [1000,1500,1800,2400,3000,3500,4000],
    'Bedrooms': [2,3,3,4,4,5,5],
    'Price': [200000,250000,300000,400000,500000,600000,650000]
}

df = pd.DataFrame(data)

X = df[['Size','Bedrooms']].values
y = df['Price'].values


# ----------------------------------
# Step 2: Define Tree Node
# ----------------------------------
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


# ----------------------------------
# Step 3: Variance Function
# ----------------------------------
def variance(y):
    return np.var(y)


# ----------------------------------
# Step 4: Find Best Split
# ----------------------------------
def best_split(X, y):
    best_feature = None
    best_threshold = None
    best_variance_reduction = -1

    parent_variance = variance(y)

    n_samples, n_features = X.shape

    for feature in range(n_features):
        thresholds = np.unique(X[:, feature])

        for threshold in thresholds:
            left_mask = X[:, feature] <= threshold
            right_mask = X[:, feature] > threshold

            if sum(left_mask) == 0 or sum(right_mask) == 0:
                continue

            left_variance = variance(y[left_mask])
            right_variance = variance(y[right_mask])

            weighted_variance = (
                (sum(left_mask)/n_samples) * left_variance +
                (sum(right_mask)/n_samples) * right_variance
            )

            variance_reduction = parent_variance - weighted_variance

            if variance_reduction > best_variance_reduction:
                best_variance_reduction = variance_reduction
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold


# ----------------------------------
# Step 5: Build Tree Recursively
# ----------------------------------
def build_tree(X, y, depth=0, max_depth=3):

    if depth >= max_depth or len(y) <= 1:
        return Node(value=np.mean(y))

    feature, threshold = best_split(X, y)

    if feature is None:
        return Node(value=np.mean(y))

    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold

    left_subtree = build_tree(X[left_mask], y[left_mask], depth+1, max_depth)
    right_subtree = build_tree(X[right_mask], y[right_mask], depth+1, max_depth)

    return Node(feature, threshold, left_subtree, right_subtree)


# ----------------------------------
# Step 6: Prediction Function
# ----------------------------------
def predict(sample, tree):

    if tree.value is not None:
        return tree.value

    if sample[tree.feature] <= tree.threshold:
        return predict(sample, tree.left)
    else:
        return predict(sample, tree.right)


# ----------------------------------
# Step 7: Train Model
# ----------------------------------
tree = build_tree(X, y, max_depth=3)

print("Decision Tree Regression Model Built Successfully")


# ----------------------------------
# Step 8: Predict New Sample
# ----------------------------------
new_house = np.array([2200, 3])  # Size=2200 sq ft, Bedrooms=3
prediction = predict(new_house, tree)

print("\nNew Sample (Size=2200, Bedrooms=3)")
print("Predicted Price:", prediction)

Decision Tree Regression Model Built Successfully

New Sample (Size=2200, Bedrooms=3)
Predicted Price: 400000.0


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss,
    mean_absolute_error, mean_squared_error, r2_score
)

# ---------------------------
# 1️⃣ Classification Dataset
# ---------------------------
print("=== Classification Metrics (Play Tennis) ===\n")

# Original dataset
df_class = pd.DataFrame({
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast',
                'Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool',
                    'Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal',
                 'High','Normal','Normal','Normal','High','Normal','High'],
    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong','Strong',
             'Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes',
                   'No','Yes','Yes','Yes','Yes','Yes','No']
})

# Encode categorical features manually for metrics calculation
mapping = {'Yes':1, 'No':0}
y_true_class = df_class['PlayTennis'].map(mapping).values

# For demonstration, we predict using a simple rule-based mapping like our ID3 tree
# Tree from previous ID3 output
def predict_playtennis(row):
    if row['Outlook'] == 'Overcast':
        return 1
    elif row['Outlook'] == 'Sunny':
        return 1 if row['Humidity']=='Normal' else 0
    elif row['Outlook'] == 'Rain':
        return 1 if row['Wind']=='Weak' else 0

y_pred_class = df_class.apply(predict_playtennis, axis=1).values

# Confusion matrix and metrics
print("Confusion Matrix:\n", confusion_matrix(y_true_class, y_pred_class))
print("Accuracy:", accuracy_score(y_true_class, y_pred_class))
print("Precision:", precision_score(y_true_class, y_pred_class))
print("Recall:", recall_score(y_true_class, y_pred_class))
print("F1 Score:", f1_score(y_true_class, y_pred_class))

# For ROC-AUC and log_loss, we need probabilities
# We will assign 1 for predicted Yes, 0 for predicted No
y_prob_class = y_pred_class  # simplified probability for illustration
print("ROC-AUC:", roc_auc_score(y_true_class, y_prob_class))
print("Log Loss:", log_loss(y_true_class, y_prob_class))

print("\n\n=== Regression Metrics (House Price) ===\n")

# ---------------------------
# 2️⃣ Regression Dataset
# ---------------------------
df_reg = pd.DataFrame({
    'Size':[1000,1500,1800,2400,3000,3500,4000],
    'Bedrooms':[2,3,3,4,4,5,5],
    'Price':[200000,250000,300000,400000,500000,600000,650000]
})

X_reg = df_reg[['Size','Bedrooms']].values
y_true_reg = df_reg['Price'].values

# Simple regression prediction using our ID3 regression tree logic
# For simplicity, assume threshold=2400 for Size split at root
y_pred_reg = []
for i in range(len(df_reg)):
    if df_reg['Size'][i] <= 2400:
        y_pred_reg.append(np.mean([200000,250000,300000,400000]))  # left leaf mean
    else:
        y_pred_reg.append(np.mean([500000,600000,650000]))          # right leaf mean
y_pred_reg = np.array(y_pred_reg)

# Regression metrics
mae = mean_absolute_error(y_true_reg, y_pred_reg)
mse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_reg, y_pred_reg)

print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R-Squared (R2 Score):", r2)



=== Classification Metrics (Play Tennis) ===

Confusion Matrix:
 [[5 0]
 [0 9]]
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0
Log Loss: 2.2204460492503136e-16


=== Regression Metrics (House Price) ===

Mean Absolute Error (MAE): 59523.80952380952
Mean Squared Error (MSE): 4791666666.666666
Root Mean Squared Error (RMSE): 69221.86552431728
R-Squared (R2 Score): 0.8172827496757458
